In [3]:
import pandas as pd
import numpy as np

In [4]:
# 1. Load the variant info
pvar = pd.read_csv('synthetic_v1_Afr_chr22only.pvar', sep='\t', comment='#', 
                   names=['CHROM', 'POS', 'ID', 'REF', 'ALT'])

pi = 0.005  # 0.5%
K = len(pvar)
num_causal = int(np.round(K * pi))

# 3. Sample True Causal Set (T)
true_causal_df = pvar.sample(n=num_causal, random_state=42)
true_causal_ids = true_causal_df['ID'].tolist()

# 4. Assign Effect Sizes (Beta)
# Under a standard infinitesimal model, beta ~ N(0, h2/num_causal)
# Note: In real data, you'd scale by genotype variance, but for 
# simulation, a normal distribution is the standard starting point.
betas = np.random.normal(0, 1, num_causal)

# 5. Save the causal effect file for PLINK
causal_effects = pd.DataFrame({
    'ID': true_causal_ids,
    'ALT': true_causal_df['ALT'],
    'BETA': betas
})
causal_effects.to_csv('true_causal_effects.txt', sep='\t', index=False, header=False)

# Save just the IDs for later "Oracle" sets
with open('true_causal_ids.txt', 'w') as f:
    for snp in true_causal_ids:
        f.write(f"{snp}\n")

In [10]:
## generate phenotype
scores = pd.read_csv('raw_genetic_scores.sscore', sep='\t')
gi = scores['NAMED_ALLELE_DOSAGE_SUM']
vg = np.var(gi)
h2 = 0.1

ve = ((1 - h2) * vg) / h2
epsilon = np.random.normal(0, np.sqrt(ve), len(gi))

scores['y'] = gi + epsilon
scores[['#FID', 'IID', 'y']].to_csv('final_pheno.pheno', sep='\t', index=False)

In [23]:
non_causal_ids = pvar[~pvar['ID'].isin(true_causal_ids)]['ID'].tolist()
true_causal_ids = causal_effects['ID'].tolist()

qs = [1.0, 0.8, 0.6, 0.4, 0.2, 0.0]

for q in qs:
    # Number of SNPs to take from the True set T
    n_true = int(np.round(num_causal * q))
    # Number of SNPs to take from the non-causal set (Noise)
    n_noise = num_causal - n_true
    
    # Construct Set P
    P_true = np.random.choice(list(true_causal_ids), n_true, replace=False)
    P_noise = np.random.choice(non_causal_ids, n_noise, replace=False)
    P_final = np.concatenate([P_true, P_noise])
    annot = pvar.copy()
    annot['score'] = annot['ID'].apply(lambda x: 1 if x in P_final else 0)
    annot.to_csv(f'set_P_q{int(q*100)}.tsv', sep='\t', index=False)

In [4]:
pheno = pd.read_csv('final_pheno.pheno', sep='\t')

In [5]:
pca = pd.read_csv('afr_pca.eigenvec', sep='\t')

In [6]:
merged = pheno.merge(pca, on=['#FID', 'IID'])

In [7]:
split = pd.read_csv('[INSERT_SPLIT_ASSIGNMENT]', sep='\t')

In [9]:
merged = merged.merge(split, on=['#FID', 'IID'])

In [11]:
merged.to_csv('final_pheno.pheno', sep='\t', index=False)